# Unit 3 — Multimedia and Interactive Visualization (UE24CS342AA9)
## Activity Notebook: The Tumour-Board Briefing

### Scenario

You are the visualization engineer in the **3-D Imaging Lab at City General Hospital**. Tomorrow morning the
neuro-oncology **tumour board** (a neurosurgeon, a radiologist and an oncologist) will review a patient's head CT
(`unit3_ppt3_head_ct_phantom.csv`). The lab head forwards you four requests:

1. **IT department:** *"The new scanner protocol is 512 × 512 × 300 at 0.5 × 0.5 × 1.0 mm. How much storage per scan,
   and how many samples does one volume-rendered frame cost?"*
2. **Radiologist:** *"Show me the lesion in all three planes, at its centre, with the anatomy undistorted."*
3. **Neurosurgeon:** *"I need the tumour volume in mL — and I want to know how much that number depends on the
   threshold you picked."*
4. **Oncologist:** *"Your last 3-D render showed a beautiful skull and nothing else. I need to see the tumour and its
   relation to the ventricles."*

Cells marked **`# TODO`** are for you to complete. Markdown cells marked **Reflection** are for you to answer in your
own words. The ground-truth `labels` array may be used **only to evaluate** your results, never to find the tumour.

## Part 0 — Setup (given)

Run this cell as-is. It loads the volume `vol[z, y, x]`, the evaluation-only `labels`, the voxel spacing, and two
helpers: `make_tf(points)` builds a transfer function from `(hu, (r, g, b), alpha)` control points, and
`render(volume, tf, axis)` ray-casts the volume with front-to-back compositing (resampled at 2× density).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage

np.random.seed(0)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

ct = pd.read_csv("unit3_ppt3_head_ct_phantom.csv")
NZ, NY, NX = ct.z.max() + 1, ct.y.max() + 1, ct.x.max() + 1
vol = ct.intensity_hu.values.reshape(NZ, NY, NX)        # V[z, y, x]  (z = feet->head, y = front->back, x = left->right)
labels = ct.label.values.reshape(NZ, NY, NX)            # ground truth: 0 air, 1 scalp, 2 bone, 3 brain, 4 CSF, 5 tumour
SPACING = (4.0, 3.0, 3.0)                               # voxel size in mm along (z, y, x)
VOXEL_MM3 = np.prod(SPACING)
ASPECT = SPACING[0] / SPACING[2]                        # use when z is on a screen axis

def make_tf(points):
    # points: list of (hu, (r, g, b), alpha) control points -> transfer function T(s) -> (C, alpha)
    hu = np.array([p[0] for p in points], float)
    rgb = np.array([p[1] for p in points], float)
    alpha = np.array([p[2] for p in points], float)
    def tf(s):
        c = np.stack([np.interp(s, hu, rgb[:, k]) for k in range(3)], axis=-1)
        return c, np.interp(s, hu, alpha)
    return tf

def render(volume, tf, axis=1, upsample=2):
    # Orthographic ray casting with front-to-back compositing; rays travel along `axis`.
    # upsample=2 resamples the volume trilinearly at twice the density (smoother, fewer aliasing artifacts).
    v = ndimage.zoom(volume.astype(float), upsample, order=1) if upsample > 1 else volume
    v = np.moveaxis(v, axis, 0)
    C = np.zeros(v.shape[1:] + (3,)); A = np.zeros(v.shape[1:])
    for sample in v:                                    # one sample per voxel step along every ray at once
        c, a = tf(sample)
        C += ((1 - A) * a)[..., None] * c
        A += (1 - A) * a
    return C, A

print(f"vol {vol.shape} (z, y, x), voxel {SPACING} mm, {VOXEL_MM3:.0f} mm^3 per voxel")

## Task 1 — The IT Department's Numbers (Request #1)

**Real-world usage:** PACS storage and GPU sizing for 3-D workstations are budgeted from exactly these figures.

**Why this technique:** Voxel count = Nx·Ny·Nz (Slides 7, 12); each ray samples the volume at successive positions
(Slide 22), so one frame costs ≈ rays × samples per ray; early ray termination skips samples once α_acc → 1
(Slide 27).

**TODO:**
1. Number of voxels in a 512 × 512 × 300 scan, and its size in MB at 16 bits per voxel.
2. Physical size of the scanned region in mm, and the volume of one voxel in mm³ (0.5 × 0.5 × 1.0 mm).
3. Samples needed for ONE front-view rendered frame (one ray per x–z pixel, one sample per voxel step in y), and
   for a 360° rotation video of 120 frames.
4. If early ray termination stops rays after 35% of their samples on average, how many samples does the 120-frame
   video need instead?

In [ ]:
# TODO 1
n_voxels = 512 * 512 * 300
scan_mb = n_voxels * 2 / (1024 ** 2)  # 16 bits = 2 bytes per voxel

# TODO 2
extent_mm = (512 * 0.5, 512 * 0.5, 300 * 1.0)  # (x, y, z)
voxel_mm3 = 0.5 * 0.5 * 1.0

# TODO 3
samples_per_frame = 512 * 300 * 512  # x-z rays, 512 samples along y
samples_video = samples_per_frame * 120

# TODO 4: 35% of the samples are processed before termination
samples_video_ert = samples_video * 0.35

print(f"{n_voxels:,} voxels = {scan_mb:.1f} MB | extent {extent_mm} mm | voxel {voxel_mm3} mm^3")
print(f"samples/frame {samples_per_frame:,} | 120 frames {samples_video:,} | with early termination {samples_video_ert:,.0f}")

**Your answer:**

Ignoring anisotropic spacing would distort anatomy because voxels are not physically equal in all directions. In coronal and sagittal views, structures would appear stretched or compressed, so their shape and proportions would be misleading. Measurements such as lesion dimensions and distances would also be inaccurate because the software would treat the 1.0 mm slice spacing as if it were 0.5 mm.

## Task 2 — Find the Lesion and Show It in Three Planes (Request #2)

**Real-world usage:** Every radiology workstation opens on a 3-plane MPR with linked cross-hairs.

**Why this technique:** **Orthogonal slices / MPR** give complementary views of the same structure (Slides 15–17);
a **window/level** setting is the slice-display transfer function; the correct **voxel aspect ratio** keeps anatomy
undistorted (Slide 9).

**TODO:**
1. Find the lesion *without* `labels`: take voxels with 55 < HU < 90 that lie **inside the brain** — use
   `inside = ndimage.binary_fill_holes(vol > 300) & ~(vol > 300)` and erode it 3 times
   (`ndimage.binary_erosion(..., iterations=3)`) to avoid the partial-volume shell under the skull.
2. Keep the largest connected component (`ndimage.label`) and compute its centroid (z, y, x) as integers.
3. Plot axial, coronal and sagittal slices through that centroid with cross-hairs, a brain window (W = 80, L = 40),
   and `aspect=ASPECT` for the two views that contain z.
4. Evaluation only: print how far (in mm) your centroid is from the centroid of `labels == 5`.

In [ ]:
def window(a, W, L):
    return np.clip((a - (L - W / 2)) / W, 0, 1)

# TODO 1: intensity range inside the eroded skull cavity
skull = vol > 300
inside = ndimage.binary_fill_holes(skull) & ~skull
inside = ndimage.binary_erosion(inside, iterations=3)
candidates = (vol > 55) & (vol < 90) & inside

# TODO 2: largest connected component and its centroid
components, n_components = ndimage.label(candidates)
component_sizes = np.bincount(components.ravel())
component_sizes[0] = 0
largest_label = int(np.argmax(component_sizes))
lesion = components == largest_label
cz, cy, cx = np.round(np.argwhere(lesion).mean(axis=0)).astype(int)

# TODO 3: linked orthogonal MPR views
fig, axes = plt.subplots(1, 3, figsize=(14, 4.6))
brain_window = lambda a: window(a, 80, 40)
axes[0].imshow(brain_window(vol[cz]), cmap="gray", origin="lower")
axes[0].axhline(cy, color="cyan"); axes[0].axvline(cx, color="yellow")
axes[0].set_title(f"Axial z={cz}")
axes[1].imshow(brain_window(vol[:, cy, :]), cmap="gray", origin="lower", aspect=ASPECT)
axes[1].axhline(cz, color="magenta"); axes[1].axvline(cx, color="yellow")
axes[1].set_title(f"Coronal y={cy}")
axes[2].imshow(brain_window(vol[:, :, cx]), cmap="gray", origin="lower", aspect=ASPECT)
axes[2].axhline(cz, color="magenta"); axes[2].axvline(cy, color="cyan")
axes[2].set_title(f"Sagittal x={cx}")
for ax in axes: ax.set_xlabel("voxel position")
axes[0].set_ylabel("voxel position")
plt.suptitle("Three-plane MPR through the detected lesion centre")
plt.tight_layout(); plt.show()

# TODO 4 (evaluation only): compare with ground truth after detection
truth_centre = np.argwhere(labels == 5).mean(axis=0)
distance_mm = np.linalg.norm((np.array([cz, cy, cx]) - truth_centre) * np.array(SPACING))
print(f"Detected centre (z,y,x): {(cz, cy, cx)}")
print(f"Distance from ground-truth tumour centroid: {distance_mm:.2f} mm")

**Your answer:**

The erosion step removes the thin partial-volume shell just inside the skull. At boundaries, a voxel can contain a mixture of tissues, so its measured HU may fall into the tumour's 55–90 HU range even though it is not tumour. Brain and tumour intensity ranges can overlap, so a transfer function based only on HU cannot reliably distinguish them. Restricting the search to an eroded brain interior reduces these false candidates.

## Task 3 — Tumour Volume and Its Threshold Sensitivity (Request #3)

**Real-world usage:** Tumour volume drives treatment decisions and response assessment, so its uncertainty must be
reported.

**Why this technique:** An **isosurface/segmentation at threshold c** defines the structure — and *the chosen c
strongly influences the resulting geometry* (Slides 19–20, 31–32).

**TODO:**
1. For c in `[45, 50, 55, 60, 65, 70]`, grow the region `vol > c` from your lesion centre (connected component
   containing `(cz, cy, cx)`).
2. For each c, record the voxel count, the volume in mL (`voxels * VOXEL_MM3 / 1000`), and — evaluation only — the
   Dice overlap with `labels == 5`.
3. Plot volume (mL) vs. threshold, and mark the threshold with the best Dice.
4. Print the range of volumes you would report to the surgeon.

In [ ]:
truth = labels == 5
rows = []
for c in [45, 50, 55, 60, 65, 70]:
    components_c, _ = ndimage.label(vol > c)
    component_id = components_c[cz, cy, cx]
    region = components_c == component_id if component_id != 0 else np.zeros_like(vol, dtype=bool)
    voxel_count = int(region.sum())
    volume_ml = voxel_count * VOXEL_MM3 / 1000
    dice_score = 2 * (region & truth).sum() / (region.sum() + truth.sum())
    rows.append({"threshold_hu": c, "voxels": voxel_count,
                 "volume_ml": volume_ml, "dice_vs_truth": dice_score})

table = pd.DataFrame(rows).round(3)
display(table)

# TODO 3
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(table["threshold_hu"], table["volume_ml"], marker="o", linewidth=2)
best_row = table.loc[table["dice_vs_truth"].idxmax()]
ax.scatter(best_row["threshold_hu"], best_row["volume_ml"], color="red", zorder=3,
           label=f"Best Dice: {best_row['threshold_hu']} HU")
ax.set_xlabel("Intensity threshold (HU)")
ax.set_ylabel("Connected-component volume (mL)")
ax.set_title("Tumour volume sensitivity to threshold")
ax.grid(alpha=0.25); ax.legend()

plt.tight_layout(); plt.show()

# TODO 4
volume_min = table["volume_ml"].min()
volume_max = table["volume_ml"].max()
print(f"Threshold-dependent volume range: {volume_min:.2f}–{volume_max:.2f} mL")
print(f"Best Dice threshold: {best_row['threshold_hu']:.0f} HU (Dice = {best_row['dice_vs_truth']:.3f})")

**Your answer:**

Without ground truth, I would select a threshold using radiologist review, consistency across adjacent slices, and agreement with other imaging or segmentation evidence rather than choosing the value that produces the most convenient volume. I would compare several clinically plausible thresholds and report the resulting volume interval, not just one number. A sensitivity plot with threshold on the x-axis and volume on the y-axis makes the dependence visible. The selected threshold and its rationale should be documented, and the estimate should be treated as uncertain until reviewed by the clinical team.

## Task 4 — A 3-D Render That Shows the Tumour (Request #4)

**Real-world usage:** Pre-operative 3-D views show a lesion in context with critical structures (here the ventricles).

**Why this technique:** **Direct volume rendering** with a designed **transfer function** — colour to *identify*
materials, opacity to *reveal or suppress* them (Slides 21–27); occlusion is the main enemy (Slide 28).

**TODO:**
1. Render the given `bone_tf` (front view, `axis=1`) — this is the "beautiful skull" the oncologist complained about.
2. Design your own `tumour_tf` with `make_tf([...])` so that: air and soft tissue are fully transparent, bone is
   *almost* transparent (α ≤ 0.01, still gives context), CSF (≈ 5–20 HU) is blue and faint, and the tumour range
   is red. Remember that opacity **accumulates over every sample along a ray**: a thick structure with α ≈ 0.1
   per sample becomes nearly opaque, while a thin one stays faint — use that to your advantage.
3. Render your TF from the front (`axis=1`) **and** from above (`axis=0`), side by side with the bone render.
4. Give the figure a conclusion-style title.

In [ ]:
bone_tf = make_tf([(-1000, (0, 0, 0), 0), (200, (0, 0, 0), 0), (400, (1, 0.95, 0.85), 0.3), (1500, (1, 1, 1), 0.6)])

# TODO 2: transparent air/soft tissue, faint blue CSF, red tumour, nearly transparent bone
tumour_tf = make_tf([
    (-1100, (0, 0, 0), 0.0),
    (4,     (0, 0, 0), 0.0),
    (5,     (0.15, 0.45, 1.0), 0.012),
    (20,    (0.15, 0.45, 1.0), 0.012),
    (35,    (0, 0, 0), 0.0),
    (50,    (0, 0, 0), 0.0),
    (55,    (1.0, 0.08, 0.08), 0.055),
    (90,    (1.0, 0.08, 0.08), 0.055),
    (110,   (0, 0, 0), 0.0),
    (200,   (0, 0, 0), 0.0),
    (300,   (1.0, 0.95, 0.85), 0.005),
    (1000,  (1.0, 1.0, 1.0), 0.005)
])

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
# TODO 1 & 3: bone render, tumour TF front view, tumour TF top view
bone_front, _ = render(vol, bone_tf, axis=1)
tumour_front, _ = render(vol, tumour_tf, axis=1)
tumour_top, _ = render(vol, tumour_tf, axis=0)
axes[0].imshow(np.clip(bone_front, 0, 1), origin="lower", aspect=ASPECT)
axes[0].set_title("Bone preset: skull dominates")
axes[1].imshow(np.clip(tumour_front, 0, 1), origin="lower", aspect=ASPECT)
axes[1].set_title("Tumour-focused TF: front")
axes[2].imshow(np.clip(tumour_top, 0, 1), origin="lower")
axes[2].set_title("Tumour-focused TF: top")
for ax in axes: ax.axis("off")

# TODO 4: suptitle
plt.suptitle("A tumour-focused transfer function reveals the lesion and ventricular context")
plt.tight_layout(); plt.show()

**Your answer:**

The red rim is caused by intensity overlap and partial-volume effects: some non-tumour voxels near the skull have HU values similar to the tumour. A transfer function maps intensity to colour and opacity, not anatomical identity, so it cannot separate two tissues that share similar values. Segmentation can remove the rim by using spatial or anatomical constraints to identify the tumour region, then displaying that label as an overlay or surface.

**Your final note:**

The detected lesion is near the centre of the brain volume at approximately voxel location (z=28, y=40, x=41), and the threshold experiment gives a volume range of approximately 6.12–19.26 mL. I recommend opening the meeting with the three-plane MPR because linked axial, coronal and sagittal slices locate the lesion while exposing internal anatomy without the occlusion of a single 3-D projection. The threshold-sensitivity plot should accompany the volume estimate so the board can see that the measurement depends on the selected HU threshold. The tumour-focused 3-D render is useful for communicating spatial context, especially the relationship to the ventricles. However, intensity overlap and partial-volume effects mean that the transfer function may also reveal non-tumour tissue, so the render should not be treated as a definitive segmentation.